# DS-02

MapReduce basics and PySpark examples.

**Web page:** <a href="https://apagyidavid.web.elte.hu/2025-2026-2/ds"
target="_blank">apagyidavid.web.elte.hu/2025-2026-2/ds</a>

<a target="_blank" href="https://colab.research.google.com/github/dapagyi/ds-web/blob/notebooks/ds-02.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MapReduce Basics

The MapReduce paradigm contains 3 phases:

1.  **Map**: split input into key-value pairs
2.  **Shuffle/Sort**: group by keys
3.  **Reduce**: aggregate values for each key

**Goal**: Efficiently perform large-scale data processing by using
parallelism across a cluster of machines.

**Cluster Architecture (Hadoop):**

Architecture divided into *master* and *worker* nodes with different
roles.

*Master Nodes*:

- NameNode: manages Hadoop Distributed File System (HDFS) metadata–for
  example controls access to files by clients
- ResourceManager: manages allocation of resources across
  cluster–schedules jobs and monitors execution
- JobHistory: stores historical information about completed jobs

*Worker Nodes*:

- DataNode: stores actual data in HDFS–manages storage and serves
  read/write requests from the file system’s clients
- NodeManager: manages execution of containers on a single node;
  monitors resource usage and reports status of node to ResourceManager
- TaskTracker: executes map and reduce tasks

**Parallel Processing in MapReduce**

<figure>
<img
src="https://apagyidavid.web.elte.hu/2025-2026-2/ds/static/map-reduce.png"
alt="Map Reduce Architecture" />
<figcaption aria-hidden="true">Map Reduce Architecture</figcaption>
</figure>

MapReduce is designed to efficiently perform large-scale data processing
by leveraging parallelism across a cluster of machines. Here’s a
detailed breakdown of how MapReduce implements parallel processing in a
cluster environment.

As we saw above:

- A *cluster* contains multiple worker nodes
- A *master node* coordinates the job, assigning map and reduce tasks to
  the workers
- Each worker node runs a TaskTracker, and the master node runs a
  JobTracker

**Step by Step**:

1.  Input Splitting: input data split into chunks (in Hadoop, 64 or 128
    MB); each split is independent and can be processed in parallel.

ex: 1 GB file with 128MB blocks –\> system creates 8 splits and
initiates 8 parallel mappers

1.  Mapping (done in parallel on multiple nodes): each mapper runs
    independently on a node and processes one data split. Mappers are
    stateless and independent, so thousands can run in parallel.

2.  Shuffle and Sort: the intermediate (key, value) pairs are shuffled
    so that *values with same key are sent to same reducer* (involving
    network transfer across nodes); data is also sorted by key.

**NOTE**: this step is very expensive; its optimization is an essential
benefit of using Hadoop/Spark.

1.  Reduce: each reducer receives a partition of keys and associated
    list of values, reducers work in parallel–output written to
    distributed storage.

Finally, the results from the reducers are written to files.

In [1]:
# Let's look at a simple example in python, without using any fancy software!

from collections import defaultdict

# Sample data: list of documents (each document is a string)
documents = [
    "hello world",
    "hello from the other side",
    "hello there",
    "world of python",
]

# MAP STEP: Break documents into words and emit (word, 1) pairs
def map_function(document):
    words = document.split()
    return [(word.lower(), 1) for word in words]


# REDUCE STEP: Sum up the counts for each word
def reduce_function(key, values):
    return (key, sum(values))


# Intermediate storage for emitted (key, value) pairs
intermediate = []

# Run map on all documents
for doc in documents:
    mapped_values = map_function(doc)
    intermediate.extend(mapped_values)

print("This is the output of the map step:")
print(intermediate)

# Shuffle and sort phase (group by key)
groups = defaultdict(list)
for key, value in intermediate:
    groups[key].append(value)

print("This is the output of the shuffle and sort step:")
print(groups)

# Run reduce on each group
results = []
for key, value_list in groups.items():
    reduced_value = reduce_function(key, value_list)
    results.append(reduced_value)

# Print final output
print("Word Count Results:")
for word, count in sorted(results):
    print(f"{word}: {count}")

This is the output of the map step:
[('hello', 1), ('world', 1), ('hello', 1), ('from', 1), ('the', 1), ('other', 1), ('side', 1), ('hello', 1), ('there', 1), ('world', 1), ('of', 1), ('python', 1)]
This is the output of the shuffle and sort step:
defaultdict(<class 'list'>, {'hello': [1, 1, 1], 'world': [1, 1], 'from': [1], 'the': [1], 'other': [1], 'side': [1], 'there': [1], 'of': [1], 'python': [1]})
Word Count Results:
from: 1
hello: 3
of: 1
other: 1
python: 1
side: 1
the: 1
there: 1
world: 2

In [2]:
# Let's look at another example, using a different reduce function.
# Here we will average instead of counting.

# Sample data: (student, score)
data = [
    ("Alice", 85),
    ("Bob", 90),
    ("Alice", 95),
    ("Bob", 80),
    ("Alice", 78),
    ("Charlie", 88),
    ("Charlie", 92),
]

# MAP STEP: Emit (student, score) pairs (already in this form)
def map_function(record):
    student, score = record
    return (student, score)


# REDUCE STEP: Compute average score per student
def reduce_function(student, scores):
    average = sum(scores) / len(scores)
    return (student, round(average, 2))


# Simulate map step
intermediate = []
for record in data:
    intermediate.append(map_function(record))

print("This is the output of the map step:")
print(intermediate)

# Shuffle and group by student
groups = defaultdict(list)
for student, score in intermediate:
    groups[student].append(score)

print("This is the output of the shuffle and sort step:")
print(groups)

# Simulate reduce step
results = []
for student, scores in groups.items():
    results.append(reduce_function(student, scores))

# Output results
print("Average Scores per Student:")
for student, avg_score in sorted(results):
    print(f"{student}: {avg_score}")

This is the output of the map step:
[('Alice', 85), ('Bob', 90), ('Alice', 95), ('Bob', 80), ('Alice', 78), ('Charlie', 88), ('Charlie', 92)]
This is the output of the shuffle and sort step:
defaultdict(<class 'list'>, {'Alice': [85, 95, 78], 'Bob': [90, 80], 'Charlie': [88, 92]})
Average Scores per Student:
Alice: 86.0
Bob: 85.0
Charlie: 90.0

## Apache Spark vs. Hadoop MapReduce

| Feature | Hadoop MapReduce | Apache Spark |
|------------------|--------------------|-----------------------------------|
| Processing Type | Disk-based | In-memory (much faster) |
| Programming Model | Rigid Map & Reduce | Flexible: Map, Reduce, Filter, Join, etc. |
| Speed | Slower (disk I/O heavy) | Much faster (RAM-centric) |
| APIs | Java-centric | Python (PySpark), Scala, Java |
| Iterative Algorithms | Inefficient | Efficient (due to caching) |

Spark introduces a more general-purpose distributed processing engine
that can run MapReduce jobs but also supports other operations like
filtering, joining, and aggregating.

In [3]:
# Install PySpark if not already installed
%pip install pyspark

# Import SparkContext
from pyspark import SparkContext

# Initialize Spark Context
sc = SparkContext("local", "WordCount").getOrCreate()

# Sample input: simulate a text file
lines = [
    "Hello world",
    "Hello from the other side",
    "Spark is fast and powerful",
    "World of big data",
]

# Create an RDD from the sample input.
# An RDD is an immutable distributed collection of elements
# of your data, partitioned across nodes in your cluster that can be operated in parallel.
rdd = sc.parallelize(lines)

# Split each line into words (flatMap), convert to lowercase, and map to (word, 1)
words = rdd.flatMap(lambda line: line.split()).map(lambda word: (word.lower(), 1))

# Reduce by key (word) to count occurrences
word_counts = words.reduceByKey(lambda a, b: a + b)

# Collect and print results
results = word_counts.collect()
for word, count in sorted(results):
    print(f"{word}: {count}")

# Stop the Spark context
sc.stop()

/home/david/code/elte/ds-web/.venv/bin/python3: No module named pip
Note: you may need to restart the kernel to use updated packages.
and: 1
big: 1
data: 1
fast: 1
from: 1
hello: 2
is: 1
of: 1
other: 1
powerful: 1
side: 1
spark: 1
the: 1
world: 2

## MapReduce Join in PySpark

Examples: joining two datasets

- Employees: emp_id, emp_name
- Departments: emp_id, department_name

*Goal*: produce pairs of the form (emp_name, department_name)

In [4]:
from pyspark import SparkContext

# Initialize SparkContext
sc = SparkContext("local", "MapReduce Join Example").getOrCreate()

# Dataset 1: (emp_id, emp_name)
employees = [
    (1, "Alice"),
    (2, "Bob"),
    (3, "Charlie"),
    (4, "Diana"),
]

# Dataset 2: (emp_id, department_name)
departments = [
    (1, "HR"),
    (2, "Engineering"),
    (3, "Sales"),
    (5, "Marketing"),  # This ID won't match any employee
]

# Create RDDs
employees_rdd = sc.parallelize(employees)
departments_rdd = sc.parallelize(departments)

# Perform inner join by emp_id
joined_rdd = employees_rdd.join(departments_rdd)

# Format the result as (emp_name, department_name)
result = joined_rdd.map(lambda x: (x[1][0], x[1][1]))

# Collect and print results
for emp_name, dept_name in result.collect():
    print(f"{emp_name}: {dept_name}")

# Stop SparkContext
sc.stop()

Bob: Engineering
Alice: HR
Charlie: Sales

## Appendix: Options for Data Loading with PySpark

**Overview**: PySpark provides a flexible ecosystem for integrating data
from various sources, including:

- Relational databases
- NoSQL databases
- Cloud-Based Storage

### Option 1: JDBC (Java Database Connectivity)

PySpark has built-in support to connect to relational databases (for
example: MySQL, PostgreSQL, SQL Server, Oracle) using JDBC.

**Pros**:

- Works with any JDBC-compatible relational database management system
- Can read specific tables or complex queries

**Cons**:

- Limited performance unless partitioning is used

In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MySQLApp") \
    .config("spark.jars", "/path/to/mysql-connector-j-8.3.0.jar") \
    .getOrCreate()

# Example download for MySQL JDBC driver
# wget https://dev.mysql.com/get/Downloads/Connector-J/mysql-connector-j-8.3.0.jar

# Then can be activated with the following script (in pseudocode, fill in with your data!)
df = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://host:port/dbname") \
    .option("dbtable", "table_name") \
    .option("user", "your_username") \
    .option("password", "your_password") \
    .load()

spark.stop()

### Option 2: PySpark Connectors for NoSQL Databases

PySpark has built-in connectors for NoSQL databases, including MongoDB,
Cassandra, HBase, Elasticsearch.

In [6]:
# Example code snippet for loading from MongoDB

df = spark.read \
    .format("mongodb") \
    .option("uri", "mongodb://127.0.0.1/db.collection") \
    .load()

### Option 3: Cloud-native Database Sources

Many cloud services (e.g., AWS, Google Cloud, Azure) offer connectors:

- Amazon Redshift: via JDBC or spark-redshift connector
- Google BigQuery: using spark-bigquery-connector
- Azure SQL Database: via JDBC

### Option 4: Exporting from the DB –\> Filesystem –\> Load into Spark

1.  Export data from the database into CSV, Parquet, JSON, or Avro
2.  Load that file into PySpark using spark.read.format(…)

In [7]:
df = spark.read.csv("path/to/exported_file.csv", header=True, inferSchema=True)

# Homework

Check
<span class="badge text-bg-warning me-1">HW/2</span><a href="https://apagyidavid.web.elte.hu/2025-2026-2/ds/notebooks/ds-hw-02" target="_blank">MapReduce</a>
for details.

**Deadline:** 2026. 04. 23. 23:59